🧩 Bloque 1 – Importación de librerías esenciales

Este bloque carga todas las librerías fundamentales para el proceso ETL:
- pandas y numpy para transformación de datos.
- sqlalchemy para conexión y escritura en SQL Server.
- pathlib y shutil para gestión de archivos.
- datetime y time para control temporal y logs.
- re y unicodedata para normalización de textos.
- Decimal para manejo preciso de decimales.

Permite trabajar con archivos Excel, limpiar datos y garantizar una integración fina con SQL Server.

In [ ]:
import pandas as pd
import re
import unicodedata
import numpy as np
from datetime import datetime
from sqlalchemy import text
import sys
import importlib
import json
from os import path
from pathlib import Path

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/itau_fraude/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path.cwd().resolve().parents[2]
FUNCTIONS_DIR = (PROJECT_ROOT / "functions").resolve()
CONFIG_DIR    = (PROJECT_ROOT / "config").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:",    CONFIG_DIR,    "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"
try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)
    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))
except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "itau_fraude", "config_itau_fraude.json")
try:
    with open(config_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()))
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")


🏗️ Bloque 2 – Configuración de la conexión a SQL Server

Este bloque define:
- Servidor, base de datos, esquema y tabla destino.
- Construcción del connection_string usando autenticación Windows.
- Motor SQLAlchemy con:
    - fast_executemany=True para inserciones masivas optimizadas.
    - pool_pre_ping=True para reconexión automática ante cortes.

Establece la base para una carga rápida y segura al Data Warehouse.

In [ ]:
# --- Parámetros de conexión ---
server   = data["server_config"]["server"]
database = data["server_config"]["database"]
schema   = data["server_config"]["schema"]
table    = data["tablas_remotas"]["tabla_principal"]

driver = "ODBC Driver 17 for SQL Server"

# ODBC connection string (pyodbc) → para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# SQLAlchemy engine
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise"
)


📂 Bloque 3 – Configuración de rutas, extensiones y selección de hoja preferida

Define:
- Carpeta donde llegan los Excel mensuales.
- Extensiones aceptadas (.xlsx).
- Lista de hojas preferidas (BBDD, BASE).
- Hoja fallback por índice.

Garantiza que el proceso siempre seleccione correctamente la hoja origen sin importar el formato del archivo.

In [ ]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "itau_fraude").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

EXTS = (".xlsx", "")  # Extensiones de archivo permitidas

PREFERRED_SHEETS = ["BBDD", "bbdd", "BASE", "base"]
FALLBACK_SHEET_INDEX = 3


🔤 Bloque 4 – Normalización de nombres de columnas

Incluye funciones para:
- Eliminar tildes y caracteres no ASCII.
- Convertir a minúsculas.
- Reemplazar espacios y símbolos por _.
- Compactar separadores repetidos.

Esto asegura encabezados uniformes y compatibles con SQL y Python para todo el flujo ETL.

🔍 Bloque 5 – Selección del archivo más reciente + Manejo de bloqueo OneDrive/Excel

Este bloque:
- Lista y ordena los archivos por fecha de modificación.
- Selecciona el archivo más reciente del partner.
- Intenta abrirlo directamente:
- Si está bloqueado (Excel/OneDrive), crea una copia temporal.
- Identifica la hoja correcta buscando coincidencia con PREFERRED_SHEETS.

Es un manejo robusto para ambientes corporativos donde los archivos suelen estar en uso.

In [ ]:
info = fun.get_latest_excel_and_sheet(
    RUTA_ARCHIVOS,
    EXTS,
    PREFERRED_SHEETS,
    fallback_sheet_index=FALLBACK_SHEET_INDEX,
    recursive=False,
    engine="openpyxl",
)

archivo_origen  = info["archivo_origen"]
excel_path_used = info["excel_path_used"]
_tmp_copy_path  = info["tmp_copy_path"]
target          = info["target_sheet"]
sheet_names     = info["sheet_names"]

print(f"Archivo: {archivo_origen.name}")
print(f"Hoja seleccionada: {target}")


📥 Bloque 6 – Lectura segura del Excel y normalización inicial

Este bloque:
- Lee la hoja objetivo con read_excel_safe(), manejando errores.
- Elimina la copia temporal si fue creada.
- Limpia encabezados y valores nulos (None, nan).
- Normaliza headers usando las funciones del Bloque 4.

El DataFrame resultante (df_raw) queda listo para ser mapeado a nombres estándar.

In [6]:
io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = fun.read_excel_safe(io, target)

if _tmp_copy_path is not None and _tmp_copy_path.exists():
    try:
        _tmp_copy_path.unlink()
        print(f"🗑️ Copia temporal eliminada: {_tmp_copy_path}")
    except Exception as e:
        print(f"⚠️ No se pudo borrar la copia temporal '{_tmp_copy_path}': {e}")

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "nan": "", "NaN": ""})

df_raw.columns = [fun.normalize_name(c) for c in df_raw.columns]

print("Encabezados normalizados:", list(df_raw.columns))

df_raw.head()

Encabezados normalizados: ['nro_cliente', 'rut', 'nombre', 'fechacobro', 'mediodepago', 'monto', 'kit', 'producto', 'prima_uf', 'vigencia', 'compania', 'direccion', 'fecha_nacimiento', 'medio_de_pago', 'xx', 'posee_ap', 'producto_sura', 'tipo_seguro', 'prima_neta_uf', 'mes', 'ano', 'mes_ano', 'prima_neta', 'uf', '39642_69994031828']


,nro_cliente,rut,nombre,fechacobro,mediodepago,monto,kit,producto,prima_uf,vigencia,...,posee_ap,producto_sura,tipo_seguro,prima_neta_uf,mes,ano,mes_ano,prima_neta,uf,39642_69994031828
0,3042438,00135546461,MERINO MANSILLA ANDR,45996,1143,10228,T4,DESEMPLEO TC,0.258,2015-10-19 00:00:00,...,1.19,DESEMPLEO TC,CESANTIA,0.217,10,2015,2015-10,8594.957983193277,39643.410852713176,0
1,2740328,00109346837,PONCE ZURITA LEONARD,45996,3810,10228,T5,DESEMPLEO TC,0.258,2015-10-28 00:00:00,...,1.19,DESEMPLEO TC,CESANTIA,0.217,10,2015,2015-10,8594.957983193277,39643.410852713176,0
2,1532354,00107322043,SANZANA CASTRO CHRIS,45996,8874,10228,T4,DESEMPLEO TC,0.258,2015-10-30 00:00:00,...,1.19,DESEMPLEO TC,CESANTIA,0.217,10,2015,2015-10,8594.957983193277,39643.410852713176,0
3,2021467,00132573484,POBLETE BARRIOS GUST,45996,2902,10228,T4,DESEMPLEO TC,0.258,2015-11-04 00:00:00,...,1.19,DESEMPLEO TC,CESANTIA,0.217,11,2015,2015-11,8594.957983193277,39643.410852713176,0
4,1470547,00136234994,MALDONADO GUTIERREZ,45996,2393,10228,T4,DESEMPLEO TC,0.258,2015-11-09 00:00:00,...,1.19,DESEMPLEO TC,CESANTIA,0.217,11,2015,2015-11,8594.957983193277,39643.410852713176,0


🧬 Bloque 7 – Mapeo de columnas origen → nombres destino

Utiliza la función pick() para detectar la columna correcta aun cuando los nombres varíen entre Exceles.
Construye df_can con todos los campos necesarios:
- Cliente, RUT, nombre, fechas, UF, montos, productos, IVA, tipo de seguro, archivo origen.
- Permite manejar columnas duplicadas, ausentes o renombradas por error.

Deja un DataFrame consolidado y homogéneo.

In [7]:
def pick(df, *names, required=False, default=None):
    for n in names:
        if n in df.columns:
            col = df[n]
            if isinstance(col, pd.DataFrame):   # columnas duplicadas
                return col.iloc[:, 0]           # tomar la primera
            return col
    if required:
        raise KeyError(f"No se encontró ninguna de las columnas: {names}")
    return pd.Series([default] * len(df), index=df.index)


# Origen → Destino
cliente             = pick(df_raw, "nro_cliente", "num_cliente")
rut                 = pick(df_raw, "rut")
nombre              = pick(df_raw, "nombre", "nombre_cliente")
fechacobro          = pick(df_raw, "fechacobro", "fecha_cobro")
codigo_pago         = pick(df_raw, "mediodepago", "medio_pago")
monto               = pick(df_raw, "monto")
kit                 = pick(df_raw, "kit")
producto            = pick(df_raw, "producto")
prima_uf            = pick(df_raw, "prima_uf")
vigencia            = pick(df_raw, "vigencia")
compania            = pick(df_raw, "compania")
fecha_nacimiento    = pick(df_raw, "fecha_nacimiento")
medio_de_pago       = pick(df_raw, "medio_de_pago")
xx                  = pick(df_raw, "xx")
posee_iva           = pick(df_raw, "posee_ap")
producto_sura       = pick(df_raw, "producto_sura")
tipo_seguro         = pick(df_raw, "tipo_seguro")
prima_neta_uf       = pick(df_raw, "prima_neta_uf")
mes                 = pick(df_raw, "mes")
ano                 = pick(df_raw, "ano")
mes_ano             = pick(df_raw, "mes_ano")
prima_neta          = pick(df_raw, "prima_neta")
uf                  = pick(df_raw, "uf")




df_can = pd.DataFrame({
    "Cliente": cliente,
    "RUT": rut,
    "NOMBRE": nombre,
    "FECHACOBRO": fechacobro,
    "CodigoPago": codigo_pago,
    "Monto": monto,
    "KIT": kit,
    "Producto": producto,
    "Prima_UF": prima_uf,
    "Vigencia": vigencia,
    "PARTNER_BANCO": compania,
    "Fecha_Nacimiento": fecha_nacimiento,
    "MediodePago": medio_de_pago,
    "xx": xx,
    "Posee_IVA": posee_iva,
    "Producto_SURA": producto_sura,
    "TIPO_SEGURO": tipo_seguro,
    "Prima_NETA_UF": prima_neta_uf,
    "MES": mes,
    "AÑO": ano,
    "MES_AÑO": mes_ano,
    "Prima_Neta": prima_neta,
    "UF": uf,
    "NOMBRE_ARCHIVO": archivo_origen.name

    
})

df_can.head()

,Cliente,RUT,NOMBRE,FECHACOBRO,CodigoPago,Monto,KIT,Producto,Prima_UF,Vigencia,...,Posee_IVA,Producto_SURA,TIPO_SEGURO,Prima_NETA_UF,MES,AÑO,MES_AÑO,Prima_Neta,UF,NOMBRE_ARCHIVO
0,3042438,00135546461,MERINO MANSILLA ANDR,45996,1143,10228,T4,DESEMPLEO TC,0.258,2015-10-19 00:00:00,...,1.19,DESEMPLEO TC,CESANTIA,0.217,10,2015,2015-10,8594.957983193277,39643.410852713176,2025.11 Bases Fraude_Noviembre 2025.xlsx
1,2740328,00109346837,PONCE ZURITA LEONARD,45996,3810,10228,T5,DESEMPLEO TC,0.258,2015-10-28 00:00:00,...,1.19,DESEMPLEO TC,CESANTIA,0.217,10,2015,2015-10,8594.957983193277,39643.410852713176,2025.11 Bases Fraude_Noviembre 2025.xlsx
2,1532354,00107322043,SANZANA CASTRO CHRIS,45996,8874,10228,T4,DESEMPLEO TC,0.258,2015-10-30 00:00:00,...,1.19,DESEMPLEO TC,CESANTIA,0.217,10,2015,2015-10,8594.957983193277,39643.410852713176,2025.11 Bases Fraude_Noviembre 2025.xlsx
3,2021467,00132573484,POBLETE BARRIOS GUST,45996,2902,10228,T4,DESEMPLEO TC,0.258,2015-11-04 00:00:00,...,1.19,DESEMPLEO TC,CESANTIA,0.217,11,2015,2015-11,8594.957983193277,39643.410852713176,2025.11 Bases Fraude_Noviembre 2025.xlsx
4,1470547,00136234994,MALDONADO GUTIERREZ,45996,2393,10228,T4,DESEMPLEO TC,0.258,2015-11-09 00:00:00,...,1.19,DESEMPLEO TC,CESANTIA,0.217,11,2015,2015-11,8594.957983193277,39643.410852713176,2025.11 Bases Fraude_Noviembre 2025.xlsx


🗂️ Bloque 8 – Canonicalización del nombre de archivo (YYYYMM)

Implementa un parser avanzado para detectar el período desde nombres caóticos:
- YYYYMM
- YYYY-MM
- MM-YYYY
- Abreviaturas con tildes: "Agosto", "AGO", "Setiembre", etc.

Luego genera un nombre canónico:
- Bases Fraude YYYYMM.xlsx

Usado para control de versiones en SQL.

In [8]:
MESES = {
    "ENERO": "01", "ENE": "01",
    "FEBRERO": "02", "FEB": "02",
    "MARZO": "03", "MAR": "03",
    "ABRIL": "04", "ABR": "04",
    "MAYO": "05", "MAY": "05",
    "JUNIO": "06", "JUN": "06",
    "JULIO": "07", "JUL": "07",
    "AGOSTO": "08", "AGO": "08",
    "SEPTIEMBRE": "09", "SETIEMBRE": "09", "SEP": "09", "SET": "09", "SEPT": "09",
    "OCTUBRE": "10", "OCT": "10",
    "NOVIEMBRE": "11", "NOV": "11",
    "DICIEMBRE": "12", "DIC": "12",
}

def _strip_accents(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))

def _extract_yyyymm_from_name(nombre: str) -> str:
    """
    Intenta extraer YYYYMM desde el nombre del archivo (sin importar la extensión).
    Cubre:
      - YYYYMM
      - YYYY[-_/ .]?MM
      - MM[-_/ .]?YYYY
      - MES(ES) + YYYY (con tildes/abreviaturas) en cualquier orden
    Lanza ValueError si no encuentra período.
    """
    stem = Path(nombre).stem
    stem_norm = _strip_accents(stem).upper()

    # 1) YYYYMM (pegado)
    m = re.search(r"(20\d{2})(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    # 2) YYYY[-_/ .]?MM
    m = re.search(r"(20\d{2})[-_/.\s]?(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    # 3) MM[-_/ .]?YYYY
    m = re.search(r"(0[1-9]|1[0-2])[-_/.\s]?(20\d{2})", stem_norm)
    if m:
        return f"{m.group(2)}{m.group(1)}"

    # 4) MES + YYYY (variantes y abreviaturas)
    # Buscamos cualquier clave de MESES presente y cualquier año 20xx
    m_year = re.search(r"(20\d{2})", stem_norm)
    if m_year:
        year = m_year.group(1)
        # Aseguramos coincidencias completas de palabra para evitar falsos positivos (e.g. "MAR" en "MARCHA")
        for mes_txt, mm in MESES.items():
            # Se agrega una búsqueda que permite un punto después de la abreviatura del mes (e.g. "SEP." -> "09")
            if re.search(rf"(?<![A-Z0-9]){mes_txt}[\.\s]?(?![A-Z0-9])", stem_norm):
                return f"{year}{mm}"

    raise ValueError(f"No pude extraer YYYYMM desde el nombre: {nombre}")

def canonicalizar_planes(nombre: str) -> str:
    """
    Devuelve un nombre canónico estandarizado: 'SC_MARSHYYYYMM.xlsx'
    Si no logra extraer el período (YYYYMM), devuelve un nombre genérico con aviso.
    """
    try:
        yyyymm = _extract_yyyymm_from_name(nombre)
        return f"Bases Fraude {yyyymm}.xlsx"
    except ValueError as e:
        print(f"⚠️ No se pudo extraer período desde el nombre '{nombre}'. "
              f"Se usará un nombre genérico. Detalle: {e}")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"Bases Fraude {timestamp}.xlsx"



nombre_canonico = canonicalizar_planes(archivo_origen.name)

print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)

df_can["NOMBRE_ARCHIVO"] = nombre_canonico

Nombre original : 2025.11 Bases Fraude_Noviembre 2025.xlsx
Nombre canónico : Bases Fraude 202511.xlsx


🗓️ Bloque 9 – Normalización avanzada de fechas mixtas

Este bloque resuelve uno de los mayores problemas de los Excel corporativos:
columnas que mezclan seriales Excel, fechas texto y valores YYYYMMDD.

La función normaliza_fecha_mixta_inplace():
- Detecta formatos mixtos automáticamente.
- Convierte seriales Excel (base 1900/1904).
- Convierte fechas texto en múltiples formatos.
- Unifica todo en formato AAAAMMDD entero (Int64).
- Identifica filas problemáticas y las registra.

Las columnas normalizadas:
- FECHACOBRO, Vigencia, Fecha_Nacimiento.

In [9]:
#Parche para solucionar problema con Fecha de Vigencia
df_can["Vigencia"] = pd.to_datetime(df_can["Vigencia"], errors="coerce")
df_can["Vigencia"] = df_can["Vigencia"].dt.strftime("%Y%m%d")
df_can["Vigencia"] = pd.to_numeric(df_can["Vigencia"], errors="coerce").astype("Int64")

In [10]:
def normaliza_fecha_mixta_inplace(df, col, usar_calendario_1904=False, max_excel_days=60000):
    """
    Normaliza df[col] mezclando:
      - Fechas texto: '05/09/2025', '2020-10-21 00:00:00', '07-04-1947'
      - Seriales Excel: 45905, 45000, etc.
      - (Opcional) Fechas tipo YYYYMMDD: 20250905, 20250905.0

    Deja la columna en formato entero YYYYMMDD (Int64).
    """
    s = df[col].astype("string").str.strip()

    s_yyyymmdd = s.str.extract(r"^(\d{8})(?:\.0)?$", expand=False)
    fechas_yyyymmdd = pd.to_datetime(s_yyyymmdd, format="%Y%m%d", errors="coerce")

    fechas_texto = pd.to_datetime(s, dayfirst=True, errors="coerce")

    nums = pd.to_numeric(s.str.replace(",", ".", regex=False), errors="coerce")

    base_1900 = pd.Timestamp("1899-12-30")
    base_1904 = pd.Timestamp("1904-01-01")
    base = base_1904 if usar_calendario_1904 else base_1900

    mask_serial = nums.notna() & nums.between(0, max_excel_days)

    fechas_num = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    if mask_serial.any():
        fechas_num.loc[mask_serial] = (
            base + pd.to_timedelta(nums.loc[mask_serial].round().astype("Int64"), unit="D")
        )

    fecha_dt = fechas_texto.combine_first(fechas_yyyymmdd).combine_first(fechas_num)

    mask_falla = fecha_dt.isna() & s.notna() & (s.str.len() > 0)
    if mask_falla.any():
        print(f"[WARN] {col}: {mask_falla.sum()} valores no se pudieron convertir correctamente. Ejemplos:")
        print(df.loc[mask_falla, col].head(10).tolist())

    df[col] = fecha_dt.dt.strftime("%Y%m%d")
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")  # Int64 permite nulos


for c in ["FECHACOBRO", "Vigencia", "Fecha_Nacimiento"]:
    normaliza_fecha_mixta_inplace(df_can, c)

print(df_can[["FECHACOBRO", "Vigencia", "Fecha_Nacimiento"]].head())


C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_26276\1052794870.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas_texto = pd.to_datetime(s, dayfirst=True, errors="coerce")
C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_26276\1052794870.py:15: UserWarning: Parsing dates in %Y%m%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  fechas_texto = pd.to_datetime(s, dayfirst=True, errors="coerce")


   FECHACOBRO  Vigencia  Fecha_Nacimiento
0    20251205  20151019          19790324
1    20251205  20151028          19690907
2    20251205  20151030          19700131
3    20251205  20151104          19771011
4    20251205  20151109          19790508


📥 Bloque 10 – Normalización del RUT + creación de DV

El bloque:
- Limpia RUT eliminando puntos, guiones y caracteres inválidos.
- Mueve automáticamente el dígito verificador a DVRUT.
- Convierte el RUT a Int64 (eliminando ceros a la izquierda).
- Normaliza el DV a mayúscula.

Es fundamental para garantizar unicidad y consistencia en SQL.

In [11]:
def normaliza_rut(df, col_rut, col_dv=None):
    """
    Limpia el RUT dejando solo números (sin ceros a la izquierda).
    Si col_dv no se pasa, lo separa automáticamente.
    """
    # Convertir a string y limpiar caracteres no numéricos ni K/k
    s = df[col_rut].astype(str).str.upper().str.strip()
    s = s.str.replace(r"[^0-9K]", "", regex=True)  # eliminar puntos, guiones, etc.

    if col_dv is None:
        df["DVRUT"] = s.str[-1]
        df[col_rut] = s.str[:-1]
        col_dv = "DVRUT"
    else:
        df[col_rut] = s

    df[col_rut] = pd.to_numeric(df[col_rut], errors="coerce").astype("Int64")

    df[col_dv] = df[col_dv].astype(str).str.upper().str.strip()

    print(f"✅ Limpieza completada para '{col_rut}'. Total filas: {len(df)}")


normaliza_rut(df_can, "RUT")

print(df_can[["RUT", "DVRUT"]].head())

✅ Limpieza completada para 'RUT'. Total filas: 13763
        RUT DVRUT
0  13554646     1
1  10934683     7
2  10732204     3
3  13257348     4
4  13623499     4


⚙️ Bloque 11 – Tipificación final y limpieza profunda del DataFrame

Aplica la tipificación final acorde al modelo SQL:
- Conversión robusta de valores vacíos.
- Detección automática de columnas INT con decimales y corrección.
- Validación de columnas críticas.

Creación de df_sql en el orden exacto requerido para SQL Server.

In [12]:
INT_COLS    = ["Cliente","RUT","DVRUT", "FECHACOBRO", "Monto", "Vigencia", "Fecha_Nacimiento", "MES", "AÑO"]
FLOAT_COLS  = ["Prima_UF", "Posee_IVA", "Prima_NETA_UF", "Prima_Neta", "UF"]
BIGINT_COLS = ["CodigoPago", "POLIZA"]
STR_COLS    = ["NOMBRE", "KIT", "Producto", "PARTNER_BANCO", "MediodePago", "xx", "Producto_SURA", "TIPO_SEGURO", "MES_AÑO", "NOMBRE_ARCHIVO"]
DATE_COLS   = []

def _to_num_series(s: pd.Series) -> pd.Series:
    s2 = s.astype(str).str.strip().replace({
        "": np.nan,
        "None": np.nan,
        "none": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
    })
    return pd.to_numeric(s2, errors="coerce")


def convert_to_datetime(df: pd.DataFrame, date_columns: list) -> pd.DataFrame:
    """
    Convierte las columnas de fechas de formato object a formato datetime (YYYY-MM-DD).
    Si la conversión falla, reemplaza con NaT (Not a Time).
    """
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=False, infer_datetime_format=True)
            # Reemplaza cualquier valor inválido con NaT (not a time) si no se puede convertir
            # Esto asegura que las fechas no válidas sean manejadas adecuadamente.
    return df

df_can = convert_to_datetime(df_can, DATE_COLS)


# BIGINT -> pandas Int64 (acepta NaN)
for c in BIGINT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# INT -> pandas Int64
for c in INT_COLS:
    if c in df_can.columns:
        s = _to_num_series(df_can[c])

        frac_mask = (s.notna()) & (s % 1 != 0)
        if frac_mask.any():
            print(f"⚠️ Columna '{c}' (INT) tiene {frac_mask.sum()} valores con decimales. Se redondearán antes de convertir a Int64.")
            # Aproximar al entero más cercano
            s = s.round(0)

        df_can[c] = s.astype("Int64")

# FLOAT -> float64
for c in FLOAT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("float64")



if "NOMBRE" in df_can.columns:
    df_can["NOMBRE"] = df_can["NOMBRE"].astype("string").str.strip().str.slice(0, 255)

if "KIT" in df_can.columns:
    df_can["KIT"] = df_can["KIT"].astype("string").str.strip().str.slice(0, 2)

if "Producto" in df_can.columns:
    df_can["Producto"] = df_can["Producto"].astype("string").str.strip().str.slice(0, 255)

if "PARTNER_BANCO" in df_can.columns:
    df_can["PARTNER_BANCO"] = df_can["PARTNER_BANCO"].astype("string").str.strip().str.slice(0, 255)

if "MediodePago" in df_can.columns:
    df_can["MediodePago"] = df_can["MediodePago"].astype("string").str.strip().str.slice(0, 255)

if "xx" in df_can.columns:
    df_can["xx"] = df_can["xx"].astype("string").str.strip().str.slice(0, 255)

if "Producto_SURA" in df_can.columns:
    df_can["Producto_SURA"] = df_can["Producto_SURA"].astype("string").str.strip().str.slice(0, 255)

if "TIPO_SEGURO" in df_can.columns:
    df_can["TIPO_SEGURO"] = df_can["TIPO_SEGURO"].astype("string").str.strip().str.slice(0, 255)

if "MES_AÑO" in df_can.columns:
    df_can["MES_AÑO"] = df_can["MES_AÑO"].astype("string").str.strip().str.slice(0, 255)

if "NOMBRE_ARCHIVO" in df_can.columns:
    df_can["NOMBRE_ARCHIVO"] = df_can["NOMBRE_ARCHIVO"].astype("string").str.strip().str.slice(0, 255)

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)


criticas = ["RUT" "Prima_UF", "Prima_NETA_UF", "POLIZA", "NOMBRE_ARCHIVO"]
presentes = [c for c in criticas if c in df_can.columns]
if presentes:
    print("\n🔎 Nulos en columnas críticas:")
    for c in presentes:
        print(f" - {c}: {int(df_can[c].isna().sum())} nulos")

cols_sql = [
    "Cliente", "RUT", "DVRUT", "NOMBRE", "FECHACOBRO", "CodigoPago", "Monto", "KIT", "Producto",
    "Prima_UF",	"Vigencia", "PARTNER_BANCO", "Fecha_Nacimiento", "MediodePago", "xx",	"Posee_IVA",
    "Producto_SURA", "TIPO_SEGURO",	"Prima_NETA_UF", "MES",	"AÑO",	"MES_AÑO",	"Prima_Neta", "UF", "NOMBRE_ARCHIVO"
]

df_sql = df_can[[c for c in cols_sql if c in df_can.columns]].copy()

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

✅ dtypes DESPUÉS:
 Cliente                      Int64
RUT                          Int64
NOMBRE              string[python]
FECHACOBRO                   Int64
CodigoPago                   Int64
Monto                        Int64
KIT                 string[python]
Producto            string[python]
Prima_UF                   float64
Vigencia                     Int64
PARTNER_BANCO       string[python]
Fecha_Nacimiento             Int64
MediodePago         string[python]
xx                  string[python]
Posee_IVA                  float64
Producto_SURA       string[python]
TIPO_SEGURO         string[python]
Prima_NETA_UF              float64
MES                          Int64
AÑO                          Int64
MES_AÑO             string[python]
Prima_Neta                 float64
UF                         float64
NOMBRE_ARCHIVO      string[python]
DVRUT                        Int64
dtype: object

🔎 Nulos en columnas críticas:
 - Prima_NETA_UF: 0 nulos
 - NOMBRE_ARCHIVO: 0 nulos

📦 df_sq

🧾 Bloque 12 – Validación de NOMBRE_ARCHIVO y conteo de filas por archivo

Este bloque valida:
- Que NOMBRE_ARCHIVO exista.
- Que tenga valores válidos (no vacíos).
- Cuenta cuántas filas trae cada archivo en el DataFrame.

Genera un diccionario {archivo → cantidad de filas}, usado luego para control de versiones.

In [13]:
assert "NOMBRE_ARCHIVO" in df_sql.columns, "Falta la columna 'NOMBRE_ARCHIVO' en df_sql."

df_sql["NOMBRE_ARCHIVO"] = (
    df_sql["NOMBRE_ARCHIVO"]
    .astype("string")
    .str.strip()
)


vals = [
    v for v in df_sql["NOMBRE_ARCHIVO"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'NOMBRE_ARCHIVO'.")

counts = (
    df_sql["NOMBRE_ARCHIVO"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} NOMBRE_ARCHIVO distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 NOMBRE_ARCHIVO distintos en el df_sql:
   - Bases Fraude 202511.xlsx: 13763 filas


🔄 Bloque 13 – Carga con reemplazo selectivo por archivo (ETL transaccional)

Este bloque implementa el core del proceso ETL:
- Se ejecuta una sola transacción.
- Para cada NOMBRE_ARCHIVO:
- Consulta si existe data previa en SQL.
- Si existe → elimina solo ese archivo (reemplazo controlado).
- Inserta solo las filas nuevas.
- Imprime un resumen:
    - Filas reemplazadas
    - Filas insertadas
    - Archivos nuevos vs antiguos

Es un flujo robusto tipo "upsert por archivo".

In [14]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {schema}.{table}
        WHERE NOMBRE_ARCHIVO = :nombre
    """)

    for nombre_archivo in vals:  
        
        df_sub = df_sql[df_sql["NOMBRE_ARCHIVO"] == nombre_archivo]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para NOMBRE_ARCHIVO = '{nombre_archivo}'. Se omite.")
            continue

        
        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para NOMBRE_ARCHIVO = '{nombre_archivo}' "
                  f"({existing_count} filas en {schema}.{table}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para NOMBRE_ARCHIVO = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

       
        df_sub.to_sql(
            name=table,
            con=conn,       
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para NOMBRE_ARCHIVO = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por NOMBRE_ARCHIVO:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")

🆕 No hay data previa para NOMBRE_ARCHIVO = 'Bases Fraude 202511.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 13763 filas para NOMBRE_ARCHIVO = 'Bases Fraude 202511.xlsx'.

📊 Resumen de carga por NOMBRE_ARCHIVO:
   - Bases Fraude 202511.xlsx: 13763 filas cargadas (archivo nuevo).


🗑️ Bloque 14 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [81]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")


🗑️ Archivo eliminado correctamente: C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\Itau_fraude\2025.10 Bases Fraude_Octubre 2025.xlsx
